# Spot prices: Bronze to Silver

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "silver_layer" / "cache.py").is_file():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError("Cannot locate project root")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession, functions as F
from silver_layer.spot_prices import ingest

BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
print("Project root:", PROJECT_ROOT)

Project root: /home/roland/UiS/rewrite/group14


## Run ingestion

Reads Bronze JSON, validates the daily batches, and writes Silver and quarantine Parquet datasets.

In [2]:
spark = (
    SparkSession.builder.master("local[8]")
    .appName("DAT535-Silver-Spot-Prices")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

report = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
print(report)

26/11/19 17:24:07 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/19 17:24:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 17:24:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Silver rows: 87,374
Quarantine rows: 346
{'dataset': 'spot_prices', 'status': 'success_with_quarantine', 'rows': 87374, 'quarantined_rows': 346, 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/spot_prices', 'quarantine_path': '/home/roland/UiS/rewrite/group14/data/silver/quarantine/spot_prices'}


## Inspect Silver

In [3]:
# Read the cleaned hourly prices.
silver_path = SILVER_ROOT / "spot_prices"
df = spark.read.parquet(str(silver_path))

df.printSchema()
print("Silver rows:", df.count())

# Compare the number of records across Norwegian price areas.
df.groupBy("price_area").count().orderBy("price_area").show()

# Preview the first hourly prices.
df.orderBy("start_time", "price_area").show(10, truncate=False)

# Each area should have at most one price per UTC start time.
duplicates = df.groupBy("price_area", "start_time").count().where("count > 1")
print("Duplicate Silver keys:", duplicates.count())

root
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- price_area: string (nullable = true)
 |-- nok_per_kwh: decimal(18,6) (nullable = true)
 |-- eur_per_kwh: decimal(18,6) (nullable = true)
 |-- exchange_rate: decimal(18,6) (nullable = true)

Silver rows: 87374
+----------+-----+
|price_area|count|
+----------+-----+
|       NO1|17470|
|       NO2|17494|
|       NO3|17470|
|       NO4|17470|
|       NO5|17470|
+----------+-----+

+-------------------+-------------------+----------+-----------+-----------+-------------+
|start_time         |end_time           |price_area|nok_per_kwh|eur_per_kwh|exchange_rate|
+-------------------+-------------------+----------+-----------+-----------+-------------+
|2023-12-31 23:00:00|2024-01-01 00:00:00|NO1       |0.604630   |0.053790   |11.240500    |
|2023-12-31 23:00:00|2024-01-01 00:00:00|NO2       |0.604630   |0.053790   |11.240500    |
|2023-12-31 23:00:00|2024-01-01 00:00:00|NO3       |0.332270   |0.

## Inspect quarantine

Invalid source batches are quarantined as whole days. A malformed JSON file is represented by one metadata-only row. the original file stays in Bronze.

In [4]:
quarantine_path = SILVER_ROOT / "quarantine" / "spot_prices"
q = spark.read.parquet(str(quarantine_path))
print("Quarantine rows:", q.count())

# Show which area-days were rejected and the reason for each rejection.
q.select("price_area", "source_date", "quarantine_reason").distinct().orderBy(
    "source_date", "price_area"
).show(100, truncate=False)

# Inspect a few rejected records and their original file paths.
q.select(
    "price_area",
    "source_date",
    "start_time",
    "nok_per_kwh",
    "quarantine_reason",
    "_source_file",
).orderBy("source_date", "price_area", "start_time").show(10, truncate=False)

Quarantine rows: 346
+----------+-----------+----------------------------------------------+
|price_area|source_date|quarantine_reason                             |
+----------+-----------+----------------------------------------------+
|NO1       |2024-10-27 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO2       |2024-10-27 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO3       |2024-10-27 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO4       |2024-10-27 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO5       |2024-10-27 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO1       |2024-10-28 |OVERLAPPING_INTERVAL                          |
|NO3       |2024-10-28 |OVERLAPPING_INTERVAL                          |
|NO4       |2024-10-28 |OVERLAPPING_INTERVAL                          |
|NO5       |2024-10-28 |OVERLAPPING_INTERVAL                          |
|NO1       |2025-10-26 |INVALID_INTERVAL_OR_VALUE,OVERLAPPING_INTERVAL|
|NO2       |2025-10-26 |INVALID_INTERVAL_OR

## Check negative prices

Negative electricity spot prices are valid. They should remain in Silver instead of being treated as bad data.

In [5]:
negative = df.where(F.col("nok_per_kwh") < 0)
print("Negative price rows:", negative.count())
negative.orderBy("start_time", "price_area").show(10, truncate=False)

Negative price rows: 1244
+-------------------+-------------------+----------+-----------+-----------+-------------+
|start_time         |end_time           |price_area|nok_per_kwh|eur_per_kwh|exchange_rate|
+-------------------+-------------------+----------+-----------+-----------+-------------+
|2024-04-07 07:00:00|2024-04-07 08:00:00|NO1       |-0.000120  |-0.000010  |11.611800    |
|2024-04-07 07:00:00|2024-04-07 08:00:00|NO2       |-0.000120  |-0.000010  |11.611800    |
|2024-04-07 08:00:00|2024-04-07 09:00:00|NO1       |-0.001860  |-0.000160  |11.611800    |
|2024-04-07 08:00:00|2024-04-07 09:00:00|NO2       |-0.001860  |-0.000160  |11.611800    |
|2024-04-07 09:00:00|2024-04-07 10:00:00|NO1       |-0.034950  |-0.003010  |11.611800    |
|2024-04-07 09:00:00|2024-04-07 10:00:00|NO2       |-0.034950  |-0.003010  |11.611800    |
|2024-04-07 10:00:00|2024-04-07 11:00:00|NO1       |-0.046330  |-0.003990  |11.611800    |
|2024-04-07 10:00:00|2024-04-07 11:00:00|NO2       |-0.046330  |

## Check daylight saving time

Inspect hourly prices around the 2024 spring and autumn clock changes for NO1. Norway has 23-hour and 25-hour days during these transitions. This is a visual check.

In [6]:
# Compare UTC timestamps around the two daylight saving time transitions.
for start_utc, end_utc in [
    ("2024-03-30 22:00:00", "2024-03-31 05:00:00"),
    ("2024-10-26 22:00:00", "2024-10-27 05:00:00"),
]:
    print("UTC window:", start_utc, "to", end_utc)
    (
        df.where(
            (F.col("price_area") == "NO1")
            & (F.col("start_time") >= F.lit(start_utc).cast("timestamp"))
            & (F.col("start_time") < F.lit(end_utc).cast("timestamp"))
        )
        .select("start_time", "end_time", "nok_per_kwh")
        .orderBy("start_time")
        .show(30, truncate=False)
    )

UTC window: 2024-03-30 22:00:00 to 2024-03-31 05:00:00
+-------------------+-------------------+-----------+
|start_time         |end_time           |nok_per_kwh|
+-------------------+-------------------+-----------+
|2024-03-30 22:00:00|2024-03-30 23:00:00|0.665900   |
|2024-03-30 23:00:00|2024-03-31 00:00:00|0.668010   |
|2024-03-31 00:00:00|2024-03-31 01:00:00|0.665440   |
|2024-03-31 01:00:00|2024-03-31 02:00:00|0.664030   |
|2024-03-31 02:00:00|2024-03-31 03:00:00|0.664150   |
|2024-03-31 03:00:00|2024-03-31 04:00:00|0.664970   |
|2024-03-31 04:00:00|2024-03-31 05:00:00|0.667300   |
+-------------------+-------------------+-----------+

UTC window: 2024-10-26 22:00:00 to 2024-10-27 05:00:00
+----------+--------+-----------+
|start_time|end_time|nok_per_kwh|
+----------+--------+-----------+
+----------+--------+-----------+



## Verify caching

In [7]:
cached_report = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
print(cached_report)

Spot prices already processed
{'dataset': 'spot_prices', 'status': 'cached', 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/spot_prices', 'quarantine_path': '/home/roland/UiS/rewrite/group14/data/silver/quarantine/spot_prices'}
